# 🏥 Healthcare Patient Readmission Analysis
**Author:** Miral Butt | Data Analyst | MPhil Statistics, LCWU Lahore

---

## 🎯 Project Overview
Hospital readmissions cost healthcare systems billions annually. This project analyzes patient data to:
- Identify key risk factors for 30-day readmission
- Build a predictive model to flag high-risk patients
- Provide actionable recommendations for healthcare providers

**Tools:** Python, Pandas, Scikit-learn, Matplotlib, Seaborn, Statistics

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (classification_report, confusion_matrix,
                              accuracy_score, roc_auc_score, roc_curve)
import warnings
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Set2')
print('✅ Libraries loaded!')

## 2. Create Patient Dataset

In [ ]:
np.random.seed(42)
n = 1500

age           = np.random.randint(18, 90, n)
gender        = np.random.choice(['Male','Female'], n)
los           = np.random.randint(1, 20, n)          # length of stay (days)
num_diagnoses = np.random.randint(1, 10, n)
num_meds      = np.random.randint(1, 15, n)
prev_admits   = np.random.poisson(1.2, n)
diabetes      = np.random.choice([0,1], n, p=[0.70, 0.30])
hypertension  = np.random.choice([0,1], n, p=[0.65, 0.35])
discharge_type= np.random.choice(['Home','Rehab','SNF'], n, p=[0.60,0.25,0.15])

# Readmission logic
readmit_prob = (
    0.15
    + (age > 65)       * 0.15
    + (prev_admits > 1)* 0.20
    + (num_diagnoses>5)* 0.15
    + (diabetes == 1)  * 0.10
    + (los > 10)       * 0.10
    + (discharge_type=='SNF') * 0.10
)
readmit_prob = np.clip(readmit_prob, 0, 1)
readmitted   = (np.random.rand(n) < readmit_prob).astype(int)

df = pd.DataFrame({
    'PatientID'      : [f'PAT-{i:04d}' for i in range(1,n+1)],
    'Age'            : age,
    'Gender'         : gender,
    'Length_of_Stay' : los,
    'Num_Diagnoses'  : num_diagnoses,
    'Num_Medications': num_meds,
    'Prev_Admissions': prev_admits,
    'Diabetes'       : diabetes,
    'Hypertension'   : hypertension,
    'Discharge_Type' : discharge_type,
    'Readmitted_30d' : readmitted,
})

print(f'Dataset Shape: {df.shape}')
print(f'Readmission Rate: {df["Readmitted_30d"].mean()*100:.1f}%')
df.head(8)

## 3. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('Healthcare Patient Readmission — EDA', fontsize=15, fontweight='bold')

# 1. Age distribution by readmission
df[df['Readmitted_30d']==0]['Age'].hist(ax=axes[0,0], alpha=0.6,
    label='Not Readmitted', color='#2ecc71', bins=20)
df[df['Readmitted_30d']==1]['Age'].hist(ax=axes[0,0], alpha=0.6,
    label='Readmitted', color='#e74c3c', bins=20)
axes[0,0].set_title('Age Distribution by Readmission', fontweight='bold')
axes[0,0].set_xlabel('Age'); axes[0,0].legend()

# 2. Readmission rate by discharge type
dis_rate = df.groupby('Discharge_Type')['Readmitted_30d'].mean()*100
bars = axes[0,1].bar(dis_rate.index, dis_rate.values,
                     color=['#3498db','#e67e22','#e74c3c'], edgecolor='white')
axes[0,1].set_title('Readmission Rate by Discharge Type', fontweight='bold')
axes[0,1].set_ylabel('Readmission Rate (%)')
for bar, val in zip(bars, dis_rate.values):
    axes[0,1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
                   f'{val:.1f}%', ha='center', fontweight='bold')

# 3. Readmission by diabetes & hypertension
conditions = pd.DataFrame({
    'Condition': ['No Condition','Diabetes Only','Hypertension Only','Both'],
    'Rate': [
        df[(df.Diabetes==0)&(df.Hypertension==0)]['Readmitted_30d'].mean()*100,
        df[(df.Diabetes==1)&(df.Hypertension==0)]['Readmitted_30d'].mean()*100,
        df[(df.Diabetes==0)&(df.Hypertension==1)]['Readmitted_30d'].mean()*100,
        df[(df.Diabetes==1)&(df.Hypertension==1)]['Readmitted_30d'].mean()*100,
    ]
})
colors_c = ['#2ecc71','#f39c12','#e67e22','#e74c3c']
bars2 = axes[1,0].bar(conditions['Condition'], conditions['Rate'],
                       color=colors_c, edgecolor='white')
axes[1,0].set_title('Readmission Rate by Medical Condition', fontweight='bold')
axes[1,0].set_ylabel('Readmission Rate (%)')
axes[1,0].tick_params(axis='x', rotation=15)
for bar, val in zip(bars2, conditions['Rate']):
    axes[1,0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
                   f'{val:.1f}%', ha='center', fontweight='bold', fontsize=9)

# 4. Correlation heatmap
num_cols = ['Age','Length_of_Stay','Num_Diagnoses','Num_Medications',
            'Prev_Admissions','Diabetes','Hypertension','Readmitted_30d']
corr = df[num_cols].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn',
            ax=axes[1,1], linewidths=0.5, vmin=-1, vmax=1)
axes[1,1].set_title('Feature Correlation Heatmap', fontweight='bold')

plt.tight_layout()
plt.savefig('healthcare_eda.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ EDA plots saved!')

## 4. Statistical Testing

In [ ]:
# Chi-square test: Diabetes vs Readmission
ct = pd.crosstab(df['Diabetes'], df['Readmitted_30d'])
chi2, p_val, dof, _ = stats.chi2_contingency(ct)
print('=== Statistical Tests ===')
print(f'Chi-Square (Diabetes vs Readmission):')
print(f'  chi2={chi2:.3f}, p-value={p_val:.4f}')
print(f'  → {"SIGNIFICANT" if p_val < 0.05 else "NOT significant"} (alpha=0.05)\n')

# T-test: Age difference between readmitted vs not
age_readmit = df[df['Readmitted_30d']==1]['Age']
age_not     = df[df['Readmitted_30d']==0]['Age']
t_stat, p_t = stats.ttest_ind(age_readmit, age_not)
print(f'T-Test (Age: Readmitted vs Not Readmitted):')
print(f'  Mean Age Readmitted    : {age_readmit.mean():.1f} years')
print(f'  Mean Age Not Readmitted: {age_not.mean():.1f} years')
print(f'  t={t_stat:.3f}, p-value={p_t:.4f}')
print(f'  → {"SIGNIFICANT" if p_t < 0.05 else "NOT significant"} (alpha=0.05)')

## 5. Predictive Modeling

In [ ]:
le = LabelEncoder()
df['Gender_Code']       = le.fit_transform(df['Gender'])
df['Discharge_Code']    = le.fit_transform(df['Discharge_Type'])

features = ['Age','Length_of_Stay','Num_Diagnoses','Num_Medications',
            'Prev_Admissions','Diabetes','Hypertension',
            'Gender_Code','Discharge_Code']
X = df[features]
y = df['Readmitted_30d']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y)

models = {
    'Logistic Regression'  : LogisticRegression(random_state=42),
    'Random Forest'        : RandomForestClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting'    : GradientBoostingClassifier(n_estimators=100, random_state=42),
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:,1]
    acc = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)
    results[name] = {'model':model,'pred':y_pred,'prob':y_prob,'acc':acc,'auc':auc}
    print(f'\n=== {name} ===')
    print(f'Accuracy: {acc*100:.2f}%  |  ROC-AUC: {auc:.3f}')
    print(classification_report(y_test, y_pred, target_names=['Not Readmitted','Readmitted']))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Model Evaluation', fontsize=14, fontweight='bold')

# ROC Curves
colors_roc = ['#3498db','#2ecc71','#e74c3c']
for (name, res), col in zip(results.items(), colors_roc):
    fpr, tpr, _ = roc_curve(y_test, res['prob'])
    axes[0].plot(fpr, tpr, color=col, linewidth=2,
                 label=f'{name} (AUC={res["auc"]:.3f})')
axes[0].plot([0,1],[0,1],'k--', linewidth=1)
axes[0].set_title('ROC Curves Comparison', fontweight='bold')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].legend(fontsize=9)

# Feature Importance
rf = results['Random Forest']['model']
feat_imp = pd.Series(rf.feature_importances_, index=features).sort_values(ascending=True)
feat_imp.plot(kind='barh', ax=axes[1], color='#9b59b6', edgecolor='white')
axes[1].set_title('Feature Importance (Random Forest)', fontweight='bold')
axes[1].set_xlabel('Importance Score')

plt.tight_layout()
plt.savefig('model_results.png', dpi=150, bbox_inches='tight')
plt.show()
best = max(results, key=lambda m: results[m]['auc'])
print(f'\n🏆 Best Model: {best} (AUC={results[best]["auc"]:.3f})')

## 6. Conclusions & Clinical Recommendations

### 📈 Key Risk Factors
| Risk Factor | Impact |
|---|---|
| Previous Admissions | Strongest predictor of readmission |
| Age > 65 | Significantly higher readmission risk |
| Multiple Diagnoses | Complex patients more likely to return |
| Diabetes + Hypertension | Combined conditions double the risk |
| SNF Discharge | Higher risk than home discharge |

### 💡 Clinical Recommendations
| Priority | Action |
|---|---|
| 🔴 High | Create dedicated follow-up program for patients with 2+ previous admissions |
| 🔴 High | Mandatory 7-day post-discharge call for patients aged 65+ |
| 🟡 Medium | Enhanced diabetes management education before discharge |
| 🟡 Medium | Improve SNF coordination and transition planning |
| 🟢 Low | Deploy ML model to score all patients at discharge |

### 🔮 Next Steps
- Include lab results (HbA1c, blood pressure readings)
- Time-series analysis of seasonal readmission patterns
- Deploy as clinical decision support tool